In [1]:
import numpy as np
import pandas as pd
import torch_geometric
import networkx as nx
from torch_geometric.data import Data, Dataset
from torch_geometric.datasets.planetoid import Planetoid
from torch_geometric.transforms.to_undirected import ToUndirected
import torch
import os

In [2]:
circles = {}
for file in os.listdir('/home/jrm28/fairness/subgraph_sketching-original/dataset/ego-gplus/raw/gplus/'):
    file_split = file.split('.') 
    circle = file_split[0]
    file_type = file_split[1]
    if circle not in circles:
        circles[circle] = dict()
    circles[circle][file_type] = '/home/jrm28/fairness/subgraph_sketching-original/dataset/ego-gplus/raw/gplus/' + file
   

In [3]:
circles['111091089527727420853']

{'featnames': '/home/jrm28/fairness/subgraph_sketching-original/dataset/ego-gplus/raw/gplus/111091089527727420853.featnames',
 'circles': '/home/jrm28/fairness/subgraph_sketching-original/dataset/ego-gplus/raw/gplus/111091089527727420853.circles',
 'egofeat': '/home/jrm28/fairness/subgraph_sketching-original/dataset/ego-gplus/raw/gplus/111091089527727420853.egofeat',
 'feat': '/home/jrm28/fairness/subgraph_sketching-original/dataset/ego-gplus/raw/gplus/111091089527727420853.feat',
 'edges': '/home/jrm28/fairness/subgraph_sketching-original/dataset/ego-gplus/raw/gplus/111091089527727420853.edges',
 'followers': '/home/jrm28/fairness/subgraph_sketching-original/dataset/ego-gplus/raw/gplus/111091089527727420853.followers'}

In [4]:
import numpy as np

edge_list = pd.read_csv(circles['111091089527727420853']['edges'], delimiter = ' ', header = None)

all_nodes = np.unique(edge_list.values.flatten())
node_mapper = dict(zip(all_nodes, range(len(all_nodes))))

mapping_fn = lambda x: node_mapper[x] if x in node_mapper else None

edge_list[0] = edge_list[0].map(mapping_fn)
edge_list[1] = edge_list[1].map(mapping_fn)

edge_list = edge_list.dropna()

features = pd.read_csv(circles['111091089527727420853']['feat'], delimiter = ' ', header = None)

features[0] = features[0].map(mapping_fn)
features.sort_values(by=[0], inplace=True)
features = features.dropna()

sens_attrs = list(features[2])

In [ ]:
transform = ToUndirected()

# feature 1 will be the sensitive attribute
x = torch.ones(len(sens_attrs), dtype=torch.float32).view(-1, 1)
y = torch.tensor(sens_attrs)

gplus = Data(x=x, y=y, edge_index=torch.tensor(edge_list.values.T))
gplus = transform(gplus)

In [ ]:
torch.save(gplus, '/home/jrm28/fairness/data/graphs/gplus_111091089527727420853.pt')

## EDA

In [13]:
(y == 0).sum(), (y == 1).sum()

(tensor(4344), tensor(594))

In [234]:
(y[gplus.edge_index].sum(0) == 0).sum() + (y[gplus.edge_index].sum(0) == 2).sum(), (y[gplus.edge_index].sum(0) == 1).sum(), 

(tensor(570860), tensor(524986))

In [14]:
(y[gplus.edge_index].sum(0) == 2).sum()

tensor(27462)